# 04. Create Chroma DB (TL_내과)

이 노트북은 아래 순서로 실행합니다.
1. `src/transform_tl_internal_docs.py`
2. `src/load_chunks_to_chroma.py`
3. 간단 검색 성능 테스트(Hit@k)


## 0) 경로 설정

In [12]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
INPUT_JSON = PROJECT_ROOT / 'data/processed/chroma_documents/TL_내과_통합_documents.json'
CHUNKS_JSON = PROJECT_ROOT / 'data/processed/chroma_documents/TL_내과_통합_chunks.json'
PERSIST_DIR = PROJECT_ROOT / 'data/processed/chroma_db/medical_db'
COLLECTION_NAME = 'medical_db'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('INPUT_JSON   =', INPUT_JSON)
print('CHUNKS_JSON  =', CHUNKS_JSON)
print('PERSIST_DIR  =', PERSIST_DIR)
print('COLLECTION   =', COLLECTION_NAME)

PROJECT_ROOT = D:\AI\projects\dermatology-rag-chatbot
INPUT_JSON   = D:\AI\projects\dermatology-rag-chatbot\data\processed\chroma_documents\TL_내과_통합_documents.json
CHUNKS_JSON  = D:\AI\projects\dermatology-rag-chatbot\data\processed\chroma_documents\TL_내과_통합_chunks.json
PERSIST_DIR  = D:\AI\projects\dermatology-rag-chatbot\data\processed\chroma_db\medical_db
COLLECTION   = medical_db


## 1) 변환 스크립트 실행 (`transform_tl_internal_docs.py`)

In [13]:
import subprocess
import sys

assert INPUT_JSON.exists(), f'입력 파일이 없습니다: {INPUT_JSON}'

cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'src/transform_tl_internal_docs.py'),
    '--input', str(INPUT_JSON),
    '--output', str(CHUNKS_JSON),
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('DONE:', CHUNKS_JSON)

RUN: d:\AI\projects\dermatology-rag-chatbot\.venv\Scripts\python.exe D:\AI\projects\dermatology-rag-chatbot\src\transform_tl_internal_docs.py --input D:\AI\projects\dermatology-rag-chatbot\data\processed\chroma_documents\TL_내과_통합_documents.json --output D:\AI\projects\dermatology-rag-chatbot\data\processed\chroma_documents\TL_내과_통합_chunks.json
DONE: D:\AI\projects\dermatology-rag-chatbot\data\processed\chroma_documents\TL_내과_통합_chunks.json


## 2) Chroma 적재 스크립트 실행 (`load_chunks_to_chroma.py`)

In [14]:
import importlib.util

has_chromadb = importlib.util.find_spec('chromadb') is not None
print('chromadb installed:', has_chromadb)
# 필요 시 설치: %pip install chromadb

chromadb installed: True


In [15]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'src/load_chunks_to_chroma.py'),
    '--input', str(CHUNKS_JSON),
    '--persist-dir', str(PERSIST_DIR),
    '--collection-name', COLLECTION_NAME,
    '--text-field', 'retrieval_text',
    '--reset-collection',
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

RUN: d:\AI\projects\dermatology-rag-chatbot\.venv\Scripts\python.exe D:\AI\projects\dermatology-rag-chatbot\src\load_chunks_to_chroma.py --input D:\AI\projects\dermatology-rag-chatbot\data\processed\chroma_documents\TL_내과_통합_chunks.json --persist-dir D:\AI\projects\dermatology-rag-chatbot\data\processed\chroma_db\medical_db --collection-name medical_db --text-field retrieval_text --reset-collection


CompletedProcess(args=['d:\\AI\\projects\\dermatology-rag-chatbot\\.venv\\Scripts\\python.exe', 'D:\\AI\\projects\\dermatology-rag-chatbot\\src\\load_chunks_to_chroma.py', '--input', 'D:\\AI\\projects\\dermatology-rag-chatbot\\data\\processed\\chroma_documents\\TL_내과_통합_chunks.json', '--persist-dir', 'D:\\AI\\projects\\dermatology-rag-chatbot\\data\\processed\\chroma_db\\medical_db', '--collection-name', 'medical_db', '--text-field', 'retrieval_text', '--reset-collection'], returncode=0)

## 3) 컬렉션 확인 + 간단 검색 성능 테스트

In [16]:
import chromadb

client = chromadb.PersistentClient(path=str(PERSIST_DIR))
col = client.get_collection(COLLECTION_NAME)
print('count =', col.count())

def quick_search_test(collection, testcases, k=3, verbose=True):
    """
    testcases: [(query, expected_doc_id_or_None), ...]
    - expected_doc_id를 주면 Hit@k 계산
    - None이면 결과만 출력
    """
    hits = 0
    judged = 0

    for idx, (query, expected_doc_id) in enumerate(testcases, start=1):
        res = collection.query(
            query_texts=[query],
            n_results=k,
            include=['documents', 'metadatas', 'distances'],
        )

        ids = res.get('ids', [[]])[0]
        dists = res.get('distances', [[]])[0]

        if verbose:
            print(f'\n[{idx}] query: {query}')
            for r, doc_id in enumerate(ids, start=1):
                dist = dists[r - 1] if r - 1 < len(dists) else None
                print(f'  {r}. id={doc_id} dist={dist}')

        if expected_doc_id is not None:
            judged += 1
            if expected_doc_id in ids:
                hits += 1

    hit_at_k = (hits / judged) if judged else None
    print('\nsummary:')
    print('  judged =', judged)
    print('  hits   =', hits)
    print(f'  Hit@{k} =', hit_at_k if hit_at_k is not None else 'N/A')

    return {'judged': judged, 'hits': hits, 'hit_at_k': hit_at_k}

# expected_doc_id가 확실한 케이스부터 소수로 시작하세요.
TEST_CASES = [
    ('천식의 급성 악화 응급 치료 약물은?', 'TL_내과:필수_1235'),
    ('기관지 내시경의 적응증은?', 'TL_내과:필수_1231'),
    ('저산소혈증의 원인과 감별진단은?', 'TL_내과:필수_1230'),
]

metrics = quick_search_test(col, TEST_CASES, k=3, verbose=True)
metrics

count = 1021

[1] query: 천식의 급성 악화 응급 치료 약물은?
  1. id=TL_내과:필수_1650 dist=0.8504161834716797
  2. id=TL_내과:필수_1230 dist=0.853999137878418
  3. id=TL_내과:필수_1652 dist=0.8666778802871704

[2] query: 기관지 내시경의 적응증은?
  1. id=TL_내과:필수_623741 dist=0.8236227631568909
  2. id=TL_내과:필수_624059 dist=0.8309359550476074
  3. id=TL_내과:필수_2790 dist=0.8613618612289429

[3] query: 저산소혈증의 원인과 감별진단은?
  1. id=TL_내과:필수_2790 dist=0.7163240909576416
  2. id=TL_내과:필수_624059 dist=0.7535095810890198
  3. id=TL_내과:필수_1230 dist=0.758399486541748

summary:
  judged = 3
  hits   = 1
  Hit@3 = 0.3333333333333333


{'judged': 3, 'hits': 1, 'hit_at_k': 0.3333333333333333}